# 04 Results Analysis

Use this notebook to inspect saved experiment CSV files, compare cleaning conditions, and build report-ready tables.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

RESULTS_PATH = Path('results/tables/main_experiment_results.csv')
SUMMARY_PATH = Path('results/tables/summary_results.csv')

if not RESULTS_PATH.exists():
    RESULTS_PATH = Path('results/tables/notebook_experiment_results.csv')
if not SUMMARY_PATH.exists() and RESULTS_PATH.name == 'notebook_experiment_results.csv':
    SUMMARY_PATH = Path('results/tables/notebook_experiment_results_summary.csv')

df = pd.read_csv(RESULTS_PATH)
summary = pd.read_csv(SUMMARY_PATH) if SUMMARY_PATH.exists() else None

print(f'Results path: {RESULTS_PATH}')
print(f'Result rows: {len(df):,}')
display(df.head())

## Matrix Coverage

In [ ]:
combo_cols = ['dataset', 'noise_type', 'noise_rate', 'model', 'random_state']
coverage = {
    'datasets': sorted(df['dataset'].unique()),
    'noise_types': sorted(df['noise_type'].unique()),
    'noise_rates': sorted(df['noise_rate'].unique()),
    'models': sorted(df['model'].unique()),
    'seeds': sorted(df['random_state'].unique()),
    'completed_combinations': len(df[combo_cols].drop_duplicates()),
}

display(pd.DataFrame([coverage]).T.rename(columns={0: 'value'}))
display(df.groupby(['dataset', 'condition']).size().unstack(fill_value=0))

## Overall Condition Means

In [ ]:
condition_means = (
    df.groupby('condition')
    .agg(
        accuracy_mean=('accuracy', 'mean'),
        f1_macro_mean=('f1_macro', 'mean'),
        auroc_mean=('auroc', 'mean'),
    )
    .reset_index()
    .sort_values('accuracy_mean', ascending=False)
)

display(condition_means)

## Detector Ranking

In [ ]:
detector_ranking = (
    df[df['condition'].isin(['remove_suspicious', 'relabel_suspicious'])]
    .groupby('detector')
    .agg(
        det_precision_mean=('det_precision', 'mean'),
        det_recall_mean=('det_recall', 'mean'),
        det_f1_mean=('det_f1', 'mean'),
        accuracy_mean=('accuracy', 'mean'),
    )
    .reset_index()
    .sort_values('det_f1_mean', ascending=False)
)

display(detector_ranking)

## Cleaning Gains Over Noisy Baseline

In [ ]:
base_cols = ['dataset', 'noise_type', 'noise_rate', 'model', 'random_state']

noisy = (
    df[df['condition'] == 'noisy_baseline']
    .set_index(base_cols)['accuracy']
    .rename('noisy_accuracy')
)
cleaned = (
    df[df['condition'].isin(['remove_suspicious', 'relabel_suspicious'])]
    .set_index(base_cols + ['condition', 'detector'])['accuracy']
    .rename('cleaned_accuracy')
    .reset_index()
)

gains = cleaned.join(noisy, on=base_cols)
gains['accuracy_gain'] = gains['cleaned_accuracy'] - gains['noisy_accuracy']

gain_summary = (
    gains.groupby(['condition', 'detector'])
    .agg(
        mean_gain=('accuracy_gain', 'mean'),
        median_gain=('accuracy_gain', 'median'),
        pct_positive=('accuracy_gain', lambda s: (s > 0).mean()),
    )
    .reset_index()
    .sort_values('mean_gain', ascending=False)
)

display(gain_summary)

## Accuracy Curves

In [ ]:
PLOT_DATASET = 'dry_bean'
PLOT_MODEL = 'random_forest'
PLOT_NOISE_TYPE = 'symmetric'

plot_df = df[
    (df['dataset'] == PLOT_DATASET)
    & (df['model'] == PLOT_MODEL)
    & (df['noise_type'] == PLOT_NOISE_TYPE)
].copy()

plot_df['plot_condition'] = plot_df.apply(
    lambda row: row['condition'] if row['detector'] == 'none' else f"{row['condition']} ({row['detector']})",
    axis=1,
)

plt.figure(figsize=(10, 6))
sns.lineplot(
    data=plot_df,
    x='noise_rate',
    y='accuracy',
    hue='plot_condition',
    marker='o',
    errorbar='sd',
)
plt.title(f'Accuracy vs noise rate: {PLOT_DATASET}, {PLOT_NOISE_TYPE}, {PLOT_MODEL}')
plt.xlabel('Noise rate')
plt.ylabel('Accuracy')
plt.tight_layout()

## Save Report-Friendly Tables

In [ ]:
analysis_dir = Path('results/tables/notebook_analysis')
analysis_dir.mkdir(parents=True, exist_ok=True)

condition_means.to_csv(analysis_dir / 'condition_means.csv', index=False)
detector_ranking.to_csv(analysis_dir / 'detector_ranking.csv', index=False)
gain_summary.to_csv(analysis_dir / 'cleaning_gain_summary.csv', index=False)

print(f'Saved analysis tables to {analysis_dir}')